# 02 · Generate the corpus, seal the ground truth, run the QA gates (GPU, requirements-gpu.txt)

Prerequisite: `01_filter_and_build.ipynb` ran (or `scripts/1_build_pairs.sh`).

Three runs of the generation cell, in this order:
1. `DRY_RUN = True` (default): prints two prompts per generator, loads no model, reports `would_generate`.
2. `DRY_RUN = False, LIMIT_PAIRS = 2`: smoke test, 22 generations, loads each generator once (a few minutes).
3. `DRY_RUN = False, LIMIT_PAIRS = None`: the real run, about one night. Resume-safe: rerunning skips finished ids.

The three passes (D0+D2, then D3 paraphrases, then D4 copies) run in order inside the cell. Watch
`data/generated/refusals.jsonl` and the `retried` / `gave_up` counters.

GPU memory: a kernel that has generated keeps the CUDA cache until `torch.cuda.empty_cache()` or a restart. If
`nvidia-smi` shows ~16 GB used with nothing running, restart the kernel before the next model-loading step.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.max_colwidth", 140)

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO / "src"))

# ---- parameters -------------------------------------------------------------
DRY_RUN = True            # True: print prompts only. False: generate.
MODELS = None             # e.g. {"gen-qwen"} to run one generator at a time; None = all
LIMIT_PAIRS = None        # e.g. 2 for a smoke test: all passes restricted to the first 2 pairs (22 generations)
LOAD_IN_4BIT = False      # True if a generator does not fit next to the KV cache in bf16
# ------------------------------------------------------------------------------

from witness_rag.io import load_config, read_jsonl, data_path
from witness_rag.schemas import Pair, from_dict
cfg = load_config()
cfg["generation"]["load_in_4bit"] = LOAD_IN_4BIT
DATA = data_path(cfg)
plan = read_jsonl(DATA / "production_log_plan.jsonl")
pairs = {r["pair_id"]: from_dict(Pair, r) for r in read_jsonl(DATA / "claims" / "pairs.jsonl")}
print(len(plan), "planned generations for", len(pairs), "pairs")

## 1. Generate (three passes)

In [ ]:
from witness_rag.generation.generate import generate

out = DATA / "generated" / "documents_generated.jsonl"
for levels in (["D0", "D2"], ["D3"], ["D4"]):
    stats = generate(cfg, plan, pairs, out, set(levels), MODELS, LIMIT_PAIRS, DRY_RUN)
    print(levels, stats)
import gc, torch
gc.collect(); torch.cuda.empty_cache()          # give the GPU back to the next step / other processes

## 2. Seal the ground truth
Writes `documents.jsonl` (what the method sees), `production_log.jsonl` and `clusters.jsonl` (sealed).

In [ ]:
from witness_rag.generation.clusters import finalize

summary = finalize(str(DATA / "production_log_plan.jsonl"), str(DATA / "human" / "human_log.jsonl"),
                   str(out), str(DATA / "human" / "human_docs.jsonl"), DATA)
print({k: v for k, v in summary.items() if k != "errors"})
for e in summary["errors"][:20]:
    print("error:", e)

## 3. QA gates and the dataset card

In [ ]:
from witness_rag.generation.qa_gates import run_gates
from witness_rag.generation.prompts import PromptBook
from witness_rag.io import write_jsonl

log = read_jsonl(DATA / "production_log.jsonl")
docs = {d["doc_id"]: d for d in read_jsonl(DATA / "documents.jsonl")}
card, violations = run_gates(list(pairs.values()), log, docs, cfg, PromptBook())
(DATA / "dataset_card.md").write_text(card, encoding="utf-8")
write_jsonl(DATA / "qa_violations.jsonl", violations)
display(Markdown(card))
pd.DataFrame(violations).groupby("check").size() if violations else "no violations"

## 4. Repair the pair set and extend the corpus (after the first full run)
Some of CrAM's GPT wrong answers are Wikipedia title fragments or otherwise unwritable strings ("Fulham disambiguation",
"Hawaii state", "Sophia 1774"): every generator fails the value check on them. `experiments/repair_pairs.py` marks such
pairs `excluded` (rule: title artifact, non-Latin, unit string, or >= 10 rejected `value_missing` attempts from >= 2
generators), appends the same number of fresh pairs per source with the improved import filter (ids continue after the
last one, nothing moves), refills the dev split, and lists the flagged documents of active pairs for regeneration.
The whole round is `scripts/3_repair_round.sh`: it applies the repair, rebuilds the human cells and the plan, drops the
flagged docs, generates everything missing with shifted seeds (generation is deterministic, a plain rerun would replay
the rejected attempts), and reseals. The cell below is the dry run. Pairs still unresolved afterwards are systematic refusals on famous facts:
decide per pair (keep the salvaged text, hand the doc to another generator, or exclude).

In [ ]:
!cd .. && python experiments/repair_pairs.py      # dry run: exclusions, replacements, regeneration list

## 5. Preview the contexts (no model)
How many contexts each condition yields on the dev split; errors list pairs with missing cells.

In [ ]:
from witness_rag.contexts.builder import CorpusView, build_context, iter_conditions, ContextError
from witness_rag.io import read_json
import collections

view = CorpusView.load(DATA / "claims" / "pairs.jsonl", DATA / "production_log.jsonl", DATA / "documents.jsonl")
dev = read_json(DATA / "claims" / "splits.json")["dev_pairs"]
built, errors = collections.Counter(), collections.Counter()
for cond, k in iter_conditions(cfg):
    for pid in dev:
        try:
            build_context(view, pid, cond, k, int(cfg["contexts"]["n_docs"]), int(cfg["project"]["seed"]))
            built[(cond, k)] += 1
        except ContextError:
            errors[cond] += 1
print("skipped by condition:", dict(errors))
pd.Series(built, name="contexts").sort_index()

Next: `scripts/4_reader_setup.sh` (hook check, calibrators, closed-book probe, heads, judge check), then `scripts/5_dev_run.sh`.